Import Library & Persiapan Dataset

In [1]:
import re
import pandas as pd
import nltk
from nltk.stem import PorterStemmer
from collections import defaultdict

Persiapan Dataset

In [23]:
# Mengambil dataset BBC News (Berasal dari Kaggle, di-host di raw github agar mudah diakses Colab)
url = "https://raw.githubusercontent.com/ievankaa/Tubes1IR/main/data/dataset.csv"
df = pd.read_csv(url, encoding='latin-1')

# Mengambil 100 artikel pertama sebagai corpus kita
# Dokumen ini dipastikan berbahasa Inggris
raw_documents = df['news'].head(100).tolist()

print(f"Berhasil memuat {len(raw_documents)} dokumen berita nyata.")
print("-" * 50)
print("Cuplikan Doc ID 0 (BBC News):\n", raw_documents[0][:300], "...\n")
print("Cuplikan Doc ID 1 (BBC News):\n", raw_documents[1][:300], "...")

Berhasil memuat 100 dokumen berita nyata.
--------------------------------------------------
Cuplikan Doc ID 0 (BBC News):
 China had role in Yukos split-up
 
 China lent Russia $6bn (Â£3.2bn) to help the Russian government renationalise the key Yuganskneftegas unit of oil group Yukos, it has been revealed.
 
 The Kremlin said on Tuesday that the $6bn which Russian state bank VEB lent state-owned Rosneft to help buy Yuga ...

Cuplikan Doc ID 1 (BBC News):
 Oil rebounds from weather effect
 
 Oil prices recovered in Asian trade on Tuesday, after falling in New York on milder winter weather across the US.
 
 With winter temperatures staying relatively high in the northern US, a barrel of light crude ended Monday down $1.33 to $42.12. However crude price ...


#Core Engine

In [3]:
class AdvancedSearchEngine:
    def __init__(self, documents):
        self.documents = documents
        self.stemmer = PorterStemmer() # Indexing menggunakan Porter Stemmer

        # Peran 1: Dictionary & Posting List (Inverted Index)
        self.inverted_index = defaultdict(list)
        self.vocab = set()

        # Peran 3: k-gram index (untuk wildcard)
        self.kgram_index = defaultdict(set)

        self._build_index()

    # ==========================================
    # PERAN 1: DATA & INDEXING ENGINEER
    # ==========================================
    def preprocess(self, text):
        """Tokenisasi dan Normalisasi (Case Folding & Stemming)"""
        # Case folding (lowercase) dan hilangkan tanda baca
        clean_text = re.sub(r'[^\w\s]', '', text.lower())
        tokens = clean_text.split()
        return [self.stemmer.stem(t) for t in tokens]

    def _build_index(self):
        """Membangun Inverted Index dan K-Gram Index (2-gram)"""
        temp_index = defaultdict(set)

        for doc_id, text in enumerate(self.documents):
            stemmed_tokens = self.preprocess(text)
            for token in stemmed_tokens:
                temp_index[token].add(doc_id)
                self.vocab.add(token)

                # Membangun K-gram index dengan marker '$'
                term_with_boundary = f"${token}$"
                for i in range(len(term_with_boundary) - 1):
                    bigram = term_with_boundary[i:i+2]
                    self.kgram_index[bigram].add(token)

        # Posting list HARUS terurut (Sorted List) untuk algoritma Intersection O(x+y)
        for term, doc_set in temp_index.items():
            self.inverted_index[term] = sorted(list(doc_set))

    # ==========================================
    # PERAN 2: BOOLEAN ENGINE DEVELOPER
    # ==========================================
    def _intersect(self, pL1, pL2):
        """Algoritma Intersection O(x+y) dengan pointer traversal"""
        answer = []
        i, j = 0, 0
        while i < len(pL1) and j < len(pL2):
            if pL1[i] == pL2[j]:
                answer.append(pL1[i])
                i += 1
                j += 1
            elif pL1[i] < pL2[j]:
                i += 1
            else:
                j += 1
        return answer

    def _union(self, pL1, pL2):
        """Algoritma Union untuk operasi OR"""
        answer = []
        i, j = 0, 0
        while i < len(pL1) and j < len(pL2):
            if pL1[i] == pL2[j]:
                answer.append(pL1[i])
                i += 1; j += 1
            elif pL1[i] < pL2[j]:
                answer.append(pL1[i])
                i += 1
            else:
                answer.append(pL2[j])
                j += 1
        while i < len(pL1):
            answer.append(pL1[i]); i += 1
        while j < len(pL2):
            answer.append(pL2[j]); j += 1
        return answer

    def _difference(self, pL1, pL2):
        """Algoritma Difference untuk operasi NOT (pL1 AND NOT pL2)"""
        answer = []
        i, j = 0, 0
        while i < len(pL1) and j < len(pL2):
            if pL1[i] == pL2[j]:
                i += 1; j += 1
            elif pL1[i] < pL2[j]:
                answer.append(pL1[i])
                i += 1
            else:
                j += 1
        while i < len(pL1):
            answer.append(pL1[i]); i += 1
        return answer

    def boolean_search(self, query):
        """Pemroses kueri Boolean sederhana: AND, OR, NOT (Urutan statis Kiri ke Kanan)"""
        tokens = query.split()
        if not tokens: return []

        # Ambil posting list dari kata pertama
        current_result = self.inverted_index.get(self.stemmer.stem(tokens[0].lower()), [])

        i = 1
        while i < len(tokens):
            operator = tokens[i].upper()
            if i + 1 >= len(tokens): break # Syntax error

            next_term = self.stemmer.stem(tokens[i+1].lower())
            next_postings = self.inverted_index.get(next_term, [])

            if operator == "AND":
                current_result = self._intersect(current_result, next_postings)
            elif operator == "OR":
                current_result = self._union(current_result, next_postings)
            elif operator == "NOT":
                # Asumsi sintaks "term1 NOT term2" artinya term1 AND NOT term2
                current_result = self._difference(current_result, next_postings)

            i += 2

        return current_result

    # ==========================================
    # PERAN 3: TOLERANT RETRIEVAL SPECIALIST
    # ==========================================

    # 3A. SPELLING CORRECTION (Edit Distance Dynamic Programming)
    def edit_distance_dp(self, s1, s2):
        """Menghitung Levenshtein Distance menggunakan Tabel DP [cite: 1499, 1500, 1502, 1503]"""
        m, n = len(s1), len(s2)
        dp = [[0] * (n + 1) for _ in range(m + 1)]

        for i in range(m + 1):
            for j in range(n + 1):
                if i == 0:
                    dp[i][j] = j  # Insert
                elif j == 0:
                    dp[i][j] = i  # Delete
                elif s1[i-1] == s2[j-1]:
                    dp[i][j] = dp[i-1][j-1] # Do nothing [cite: 1463, 1464]
                else:
                    dp[i][j] = 1 + min(dp[i][j-1],      # Insert
                                       dp[i-1][j],      # Delete
                                       dp[i-1][j-1])    # Replace
        return dp[m][n]

    def spell_check(self, typo_term):
        """Mencari term di vocabulary dengan edit distance terkecil"""
        typo_term = self.stemmer.stem(typo_term.lower())
        min_dist = float('inf')
        closest_word = typo_term

        for vocab_word in self.vocab:
            dist = self.edit_distance_dp(typo_term, vocab_word)
            if dist < min_dist:
                min_dist = dist
                closest_word = vocab_word
        return closest_word

    # 3B. WILDCARD SEARCH (K-Gram Index)
    def wildcard_search(self, query):
        """Mencari wildcard menggunakan intersection dari k-gram"""
        query = query.lower()
        parts = query.split('*')

        # Ekstrak k-gram (bigram) dari kueri yang mengandung wildcard
        query_bigrams = set()
        if query.startswith('*') and query.endswith('*'): # *tengah*
            core = parts[1]
            for i in range(len(core)-1): query_bigrams.add(core[i:i+2])
        elif query.endswith('*'): # awalan*
            core = f"${parts[0]}"
            for i in range(len(core)-1): query_bigrams.add(core[i:i+2])
        elif query.startswith('*'): # *akhiran
            core = f"{parts[1]}$"
            for i in range(len(core)-1): query_bigrams.add(core[i:i+2])

        # Ambil vocabulary term yang memiliki SEMUA bigram tersebut (Intersect)
        candidate_words = None
        for bg in query_bigrams:
            bg_postings = self.kgram_index.get(bg, set())
            if candidate_words is None:
                candidate_words = bg_postings
            else:
                candidate_words = candidate_words.intersection(bg_postings)

        if not candidate_words:
            return []

        # Post-filtering dengan Regex untuk membuang False Positives
        regex_pattern = "^" + query.replace("*", ".*") + "$"
        valid_words = [w for w in candidate_words if re.match(regex_pattern, w)]

        # Union semua posting list dari term yang lolos filter
        result_docs = []
        for word in valid_words:
            result_docs = self._union(result_docs, self.inverted_index.get(word, []))

        return result_docs, valid_words

# Test Cases

In [4]:
# Inisialisasi
engine = AdvancedSearchEngine(raw_documents)
print(f"Index berhasil dibangun. Ukuran Vocabulary: {len(engine.vocab)} terms\n")

Index berhasil dibangun. Ukuran Vocabulary: 4248 terms



DEMO PERAN 2: BOOLEAN ENGINE

In [5]:
print("DEMO PERAN 2: BOOLEAN ENGINE")
print("="*50)
# Kueri: dokumen yang memiliki kata "jury" DAN "said"
query_and = "jury AND said"
result_and = engine.boolean_search(query_and)
print(f"Kueri: {query_and}")
print(f"Daftar Doc ID: {result_and}")
for doc_id in result_and:
    print(f" -> Doc {doc_id}: {engine.documents[doc_id]}")

print("\n" + "="*50)

DEMO PERAN 2: BOOLEAN ENGINE
Kueri: jury AND said
Daftar Doc ID: [49]
 -> Doc 49: Worldcom ex-boss launches defence
 
 Lawyers defending former WorldCom chief Bernie Ebbers against a battery of fraud charges have called a company whistleblower as their first witness.
 
 Cynthia Cooper, WorldCom's ex-head of internal accounting, alerted directors to irregular accounting practices at the US telecoms giant in 2002. Her warnings led to the collapse of the firm following the discovery of an $11bn (Â£5.7bn) accounting fraud. Mr Ebbers has pleaded not guilty to charges of fraud and conspiracy.
 
 Prosecution lawyers have argued that Mr Ebbers orchestrated a series of accounting tricks at WorldCom, ordering employees to hide expenses and inflate revenues to meet Wall Street earnings estimates. But Ms Cooper, who now runs her own consulting business, told a jury in New York on Wednesday that external auditors Arthur Andersen had approved WorldCom's accounting in early 2001 and 2002. She said An

DEMO PERAN 3A

In [9]:
print("="*50)
print("DEMO PERAN 3A: SPELLING CORRECTION (DP Levenshtein)")
print("="*50)
typo = "juri" # Sengaja salah ketik dari 'jury'
koreksi = engine.spell_check(typo)
print(f"Kata Typo     : '{typo}'")
print(f"Hasil Koreksi : '{koreksi}'")
print(f"Doc IDs untuk '{koreksi}': {engine.boolean_search(koreksi)}")

DEMO PERAN 3A: SPELLING CORRECTION (DP Levenshtein)
Kata Typo     : 'juri'
Hasil Koreksi : 'juri'
Doc IDs untuk 'juri': [49, 65]


DEMO PERAN 3B: WILDCARD SEARCH (K-Gram Indexing)

In [7]:
print("\n" + "="*50)
print("DEMO PERAN 3B: WILDCARD SEARCH (K-Gram Indexing)")
print("="*50)
# Kueri wildcard trailing: kata yang diawali 'invest' (misal: investigation)
wildcard_query = "invest*"
docs, words = engine.wildcard_search(wildcard_query)
print(f"Kueri Wildcard: '{wildcard_query}'")
print(f"Term hasil filter K-gram: {words}")
print(f"Doc IDs Ditemukan: {docs}")
for doc_id in docs[:2]: # Tampilkan maks 2 dokumen
    print(f" -> Doc {doc_id}: {engine.documents[doc_id]}")


DEMO PERAN 3B: WILDCARD SEARCH (K-Gram Indexing)
Kueri Wildcard: 'invest*'
Term hasil filter K-gram: ['invest', 'investor', 'investig']
Doc IDs Ditemukan: [3, 4, 8, 9, 11, 14, 17, 22, 28, 29, 31, 33, 35, 36, 37, 39, 40, 45, 46, 47, 50, 51, 52, 55, 57, 59, 60, 62, 65, 70, 76, 78, 81, 82, 85, 86, 88, 90, 93, 96, 98, 99]
 -> Doc 3: $1m payoff for former Shell boss
 
 Shell is to pay $1m (Â£522,000) to the ex-finance chief who stepped down from her post in April 2004 after the firm over-stated its reserves.
 
 Judy Boynton finally left the firm on 31 December, having spent the intervening time as a special advisor to chief executive Jeroen van der Veer. In January 2004, Shell told shocked investors that its reserves were 20% smaller than previously thought. Shell said the pay-off was in line with Ms Boynton's contract. She was leaving "by mutual agreement to pursue other career opportunities", the firm said in a statement. The severance package means she keeps long-term share options, but